# 01 · Pipeline: de la base operacional a RAW y MASTER

> **Qué haces tú:** ejecutar el cuaderno entero (**Ejecutar todo**) y leer lo que sale. No hay código que escribir.

## Objetivo
1. **RAW (ingesta):** copiar las tablas de `cqi.db` a la base analítica `cafe.db` **sin tocar nada**. RAW es la copia fiel de lo que llegó.
2. **MASTER (depuración):** construir **una fila por lote catado**, con el dato corregido o apartado según reglas claras, y **medir la calidad**.

| | RAW | MASTER |
|---|---|---|
| Qué guarda | Lo que llegó, tal cual (con sus errores) | El dato depurado: una fila por lote |
| Qué hace con un error | Lo conserva | Lo corrige, lo deja nulo o aparta el lote |
| ¿Se puede perder información? | No: es el respaldo | Sí, por eso RAW sigue existiendo |

**Idempotencia.** Estos pipelines se pueden ejecutar **dos veces seguidas** sin estropear nada: si la entrada no ha cambiado, la segunda vez
no hace nada y lo dice (`sin cambios`). Lo comprobarás aquí.

### Preparación
Se carga el paquete del proyecto y se comprueba que existe la base operacional (la crea el cuaderno 00).

In [1]:
import sys
import warnings
from pathlib import Path

import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.width', 160, 'display.max_columns', 30, 'display.max_colwidth', 70)

# Carpeta del proyecto: la que contiene «sql» y «pyproject.toml». Así el cuaderno funciona se abra desde donde se abra.
RAIZ = Path.cwd().resolve()
while not ((RAIZ / 'sql').exists() and (RAIZ / 'pyproject.toml').exists()) and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))

from cafe_mlops import config, db   # el paquete del proyecto: lo mismo que ejecutan las órdenes de la terminal

print('Proyecto listo.')
print('Base operacional:', config.DB_OPERACIONAL.name, '· base analítica:', config.DB.name)

if not config.DB_OPERACIONAL.exists():
    raise FileNotFoundError('Falta la base operacional: ejecuta antes el cuaderno 00.')

from cafe_mlops import calidad, dataset, pipeline_ingesta_master, pipeline_ingesta_raw

Proyecto listo.
Base operacional: cqi.db · base analítica: cafe.db


## 1. Ingesta a RAW
`pipeline_ingesta_raw.ejecutar()` copia las siete tablas de `cqi.db` a `cafe.db` con el prefijo `raw_`, y anota en `raw_carga` **cuántas filas** y **qué huella** tenía
cada tabla. Antes de ejecutar calcula una **huella de entrada** (el contenido del origen y el código de la ingesta): si es la misma que la de la última vez, no hace nada.

In [2]:
resultado_raw = pipeline_ingesta_raw.ejecutar()

[pipeline_ingesta_raw] ejecutando (huella de entrada 7379031448d0)
  RAW: raw_paises                35 filas
  RAW: raw_entidades             26 filas
  RAW: raw_productores          621 filas
  RAW: raw_lotes              1,190 filas
  RAW: raw_analisis_verde     1,190 filas
  RAW: raw_catas              1,190 filas
  RAW: raw_notas_cata        11,900 filas


`raw_carga` es el **registro de la carga**: una fila por tabla, con sus filas y la huella (SHA-256) de su contenido. Es la evidencia de que lo copiado es lo que había.

In [3]:
con = db.conectar()
pd.read_sql('SELECT tabla, filas, substr(sha256, 1, 12) AS huella FROM raw_carga ORDER BY carga_id', con)

,tabla,filas,huella
0,paises,35,5eca0f1b47b4
1,entidades,26,e04ba79f22d8
2,productores,621,dc61f2ba59b7
3,lotes,1190,98fe55ad9e43
4,analisis_verde,1190,87c0981e374f
5,catas,1190,203203742021
6,notas_cata,11900,624c11abe7f1


**¿RAW es igual al operacional?** Contamos las filas de cada tabla en los dos sitios. La columna `igual` debe ser `True` en todas.

In [4]:
con.execute('ATTACH DATABASE ? AS cqi', (str(config.DB_OPERACIONAL),))
comparacion = pd.DataFrame([
    {'tabla': t, 'operacional': db.escalar(con, f'SELECT COUNT(*) FROM cqi.{t}'), 'RAW': db.escalar(con, f'SELECT COUNT(*) FROM raw_{t}')}
    for t in dataset.TABLAS_OPERACIONALES])
comparacion['igual'] = comparacion['operacional'] == comparacion['RAW']
con.execute('DETACH DATABASE cqi')
assert comparacion['igual'].all()
comparacion

,tabla,operacional,RAW,igual
0,paises,35,35,True
1,entidades,26,26,True
2,productores,621,621,True
3,lotes,1190,1190,True
4,analisis_verde,1190,1190,True
5,catas,1190,1190,True
6,notas_cata,11900,11900,True


### Probar la idempotencia
Ejecutamos **la misma ingesta otra vez**. La entrada no ha cambiado, así que debe decir `sin cambios` y no volver a copiar nada.

In [5]:
segunda_raw = pipeline_ingesta_raw.ejecutar()
print('\nEstado de la segunda ejecución:', segunda_raw['estado'])

[pipeline_ingesta_raw] sin cambios: la entrada es la misma que en la ultima ejecucion (7379031448d0). Usa --forzar para rehacerlo.



Estado de la segunda ejecución: sin cambios


## 2. De RAW a MASTER
`pipeline_ingesta_master.ejecutar()` construye `master_lotes` (un lote catado por fila) con estas reglas, escritas en `sql/master/02_poblar_master.sql`:
- la **fecha de cata** pasa de texto («April 4th, 2015») a `AAAA-MM-DD`, quitando espacios y saltos de línea;
- una **altitud** fuera de 100–4 000 m se deja **nula** (se conserva el valor original en `altitud_media_origen_m`);
- una **humedad de 0** se considera «no medida» y se deja nula (se conserva en `humedad_origen`);
- una cata con **total 0** es incompleta: el lote va a `master_descartes`.

Después evalúa las **reglas de calidad** y guarda el resultado en `master_calidad`.

In [6]:
resultado_master = pipeline_ingesta_master.ejecutar()

[pipeline_ingesta_master] ejecutando (huella de entrada 383d741d7a61)


  MASTER: master_lotes          1,190 filas
  MASTER: master_descartes          0 filas
  MASTER: calidad: 17 reglas, 3 alertas (R06, R07, R10)


### Qué entró y qué se apartó
La conciliación: **lotes de RAW = lotes de MASTER + descartes**. Ningún lote se pierde sin que quede anotado el motivo.
Con la versión 1 de los datos no hay ninguna cata incompleta que apartar, así que la tabla de motivos sale vacía; cuando lleguen los datos nuevos (versión 2) aparecerá una cata con total 0.

In [7]:
raw_lotes = db.escalar(con, 'SELECT COUNT(*) FROM raw_lotes')
master = db.escalar(con, 'SELECT COUNT(*) FROM master_lotes')
descartes = db.escalar(con, 'SELECT COUNT(*) FROM master_descartes')
assert raw_lotes == master + descartes
print(f'Lotes en RAW: {raw_lotes:,}  =  lotes en MASTER: {master:,}  +  descartes: {descartes}')
pd.read_sql('SELECT motivo, COUNT(*) AS lotes FROM master_descartes GROUP BY motivo', con)

Lotes en RAW: 1,190  =  lotes en MASTER: 1,190  +  descartes: 0


,motivo,lotes


## 3. Reglas de calidad
Una **regla de calidad** es una consulta que devuelve **un número**, más un criterio que dice si es aceptable. Hay tres tipos:

| Tipo | Si no se cumple | Estado |
|---|---|---|
| **duro** | El pipeline **se detiene** (seguir daría resultados falsos) | `ERROR` |
| **alerta** | Se avisa y el pipeline sigue (MASTER ya corrigió o apartó el dato) | `ALERTA` |
| **info** | Solo informa | `OK` |

La tabla `master_calidad` es la **evidencia** de que el dato se revisó. Mira qué reglas están en `ALERTA` y qué pasó con ese dato en la sección siguiente.

In [8]:
with pd.option_context('display.max_colwidth', 105, 'display.width', 200):
    display(pd.read_sql('SELECT regla, descripcion, valor, umbral, estado FROM master_calidad ORDER BY rowid', con))

,regla,descripcion,valor,umbral,estado
0,R01_conciliacion,Lotes de RAW menos (lotes de MASTER + descartes). Debe ser 0.,0.00,= 0,OK
1,R02_lotes_sin_las_diez_notas,Lotes de MASTER que no tienen exactamente 10 notas de cata en RAW.,0.00,= 0,OK
2,R03_lotes_con_varias_catas,Lotes de RAW con más de una cata (MASTER espera una por lote).,0.00,= 0,OK
3,R04_diferencia_total_menos_notas,Mayor diferencia absoluta entre Total.Cup.Points y la suma de las diez notas (la fuga: el total ES es...,0.50,informativo,OK
4,R05_fechas_de_cata_en_rango,Fechas de cata de MASTER fuera de 2009-01-01 a 2019-01-01 (exclusivo) o mal formadas.,0.00,= 0,OK
5,R06_fechas_con_espacios_o_saltos,"Fechas de cata de RAW con espacios, tabuladores o saltos de línea sobrantes (MASTER los quita).",1.00,= 0,ALERTA
6,R07_altitud_fuera_de_rango,Lotes con altitud media fuera de 100–4000 m (MASTER la deja nula).,22.00,= 0,ALERTA
7,R08_pct_altitud_nula_en_master,Porcentaje de lotes de MASTER sin altitud utilizable.,19.41,informativo,OK
8,R09_catas_con_total_cero,Catas de RAW con total 0 (cata incompleta; MASTER las descarta).,0.00,= 0,OK
9,R10_humedad_cero,Lotes con humedad 0 en RAW (parece «no medido»; MASTER la deja nula).,231.00,= 0,ALERTA


## 4. Antes y después: lo que MASTER corrigió
**Altitudes imposibles** (fuera de 100–4 000 m): en RAW están como llegaron; en MASTER, nulas.

In [9]:
pd.read_sql(f"""
    SELECT m.lote_id, r.altitud_media_m AS altitud_en_RAW, m.altitud_media_m AS altitud_en_MASTER
    FROM raw_lotes r JOIN master_lotes m ON m.lote_id = r.lote_id
    WHERE r.altitud_media_m IS NOT NULL AND (r.altitud_media_m < {config.ALTITUD_MIN_M} OR r.altitud_media_m > {config.ALTITUD_MAX_M})
    ORDER BY m.lote_id LIMIT 6""", con)

,lote_id,altitud_en_RAW,altitud_en_MASTER
0,41,12.0,None
1,42,12.0,None
2,95,1.0,None
3,253,1.0,None
4,429,1.0,None
5,487,11000.0,None


**Humedad igual a 0**: una humedad exactamente 0 no es creíble (el grano verde siempre tiene algo). MASTER la trata como **no medida** (nula) y conserva el valor original.

In [10]:
pd.read_sql("""
    SELECT lote_id, humedad_origen AS humedad_en_RAW, humedad AS humedad_en_MASTER
    FROM master_lotes WHERE humedad_origen = 0 ORDER BY lote_id LIMIT 6""", con)

,lote_id,humedad_en_RAW,humedad_en_MASTER
0,3,0.0,None
1,12,0.0,None
2,13,0.0,None
3,14,0.0,None
4,16,0.0,None
5,20,0.0,None


**Fechas con espacios o saltos de línea**: se muestran con `repr` para que se vea lo que sobraba (`\n`, espacios) y cómo queda la fecha en MASTER.

In [11]:
fechas = pd.read_sql("""
    SELECT c.lote_id, c.fecha_cata_texto AS fecha_en_RAW, m.fecha_cata AS fecha_en_MASTER
    FROM raw_catas c JOIN master_lotes m ON m.lote_id = c.lote_id
    WHERE c.fecha_cata_texto <> TRIM(c.fecha_cata_texto, char(32, 9, 10, 13)) ORDER BY c.lote_id""", con)
fechas['fecha_en_RAW'] = fechas['fecha_en_RAW'].map(repr)
fechas

,lote_id,fecha_en_RAW,fecha_en_MASTER
0,960,"'December 26th, 2013\n'",2013-12-26


## 5. Una fila de MASTER
Una fila por lote, con el año de la cata (`anio_cata`, que más adelante repartirá los lotes en el tiempo) y la puntuación total (`total_puntos`, que **solo** servirá
para calcular la respuesta `y`, nunca como variable).

In [12]:
display(pd.read_sql('SELECT * FROM master_lotes ORDER BY lote_id LIMIT 5', con))
con.close()

,lote_id,especie,pais,variedad,metodo_proceso,color,altitud_media_m,altitud_media_origen_m,humedad,humedad_origen,defectos_cat1,defectos_cat2,quakers,numero_sacos,fecha_cata,anio_cata,total_puntos
0,1,Arabica,Ethiopia,None,Washed / Wet,Green,2075.0,2075.0,0.12,0.12,0.0,0.0,0.0,300,2015-04-04,2015,90.58
1,2,Arabica,Ethiopia,Other,Washed / Wet,Green,2075.0,2075.0,0.12,0.12,0.0,1.0,0.0,300,2015-04-04,2015,89.92
2,3,Arabica,Guatemala,Bourbon,None,None,1700.0,1700.0,NaN,0.00,0.0,0.0,0.0,5,2010-05-31,2010,89.75
3,4,Arabica,Ethiopia,None,Natural / Dry,Green,2000.0,2000.0,0.11,0.11,0.0,2.0,0.0,320,2015-03-26,2015,89.00
4,5,Arabica,Ethiopia,Other,Washed / Wet,Green,2075.0,2075.0,0.12,0.12,0.0,2.0,0.0,300,2015-04-04,2015,88.83


## Resumen
- **RAW** es la copia fiel del operacional: mismas filas, misma huella, y anotado en `raw_carga`.
- **MASTER** es el dato depurado: una fila por lote; los errores se corrigen, se dejan nulos o se apartan **con motivo**.
- Las **reglas de calidad** quedan guardadas: son la evidencia de la revisión.
- Los pipelines son **idempotentes**: ejecutados dos veces seguidas, la segunda dice `sin cambios`.

**Siguiente:** el cuaderno **02** calcula las variables de cada lote (capa FS).

## Para practicar
1. ¿Qué regla de calidad detendría el pipeline si fallase y por qué es «dura»?
2. ¿Por qué MASTER conserva `humedad_origen` si ya tiene la humedad corregida?